# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 12

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. После сортировки таблицы по балансу `df_sorted = df.sort_values(by='balance', ascending=False)` аналитик заметил, что числа в крайнем левом столбце идут вразнобой (например, 14, 2, 88...).
Что представляет собой этот столбец и какой метод позволяет восстановить непрерывную нумерацию строк от 0?

**Ответ:**
Этот столбец — **индекс строк** (`df.index`), сохранивший исходные номера записей до сортировки. Чтобы восстановить последовательную нумерацию от 0, вызывается метод `df_sorted.reset_index(drop=True)`.

---

#### 1.2. Для чего в анализе данных используется атрибут `df.shape`?
Что вернет этот атрибут для таблицы из 150 клиентов и 7 столбцов, и под каким индексом кортежа (`[0]` или `[1]`) хранится количество строк?

**Ответ:**
`df.shape` возвращает размерность таблицы в виде кортежа `(150, 7)`. Количество строк хранится под индексом `[0]`, а количество столбцов — под индексом `[1]`.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["avg_item_cost"] = (df["total_amount"] / df["quantity"]).round(2)
```

**Ответ:**
Определяет фактическую среднюю стоимость единицы товара в конкретном чеке бара.

---

#### 1.4. В чем ключевая разница между фильтрацией и агрегацией строк датафрейма?
```python
# Действие А:
df[df['total_paid'] > 500]
# Действие Б:
df.groupby('game_title')['total_paid'].sum()
```
Какое действие отбирает существующие строки без изменения их содержимого, а какое — сжимает (схлопывает) множество строк в одну?

**Ответ:**
* Действие А выполняет **фильтрацию**: оно отбирает исходные строки сессий с оплатой > 500 руб., сохраняя структуру каждой строки.
* Действие Б выполняет **агрегацию**: оно объединяет строки в группы по играм и сворачивает их в одну итоговую сумму на каждую игру.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `hardware_pcs.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("hardware_pcs.csv").parent if os.path.exists("hardware_pcs.csv") else pathlib.Path("../variant_12")
df_audit = pd.read_csv(data_dir / "hardware_pcs.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Максимальная часовая ставка ПК: 234.00")

Количество строк в таблице: 30
Столбцы датафрейма: ['pc_id', 'pc_number', 'zone_name', 'hourly_rate', 'gpu', 'cpu', 'ram_gb', 'status']
Максимальная часовая ставка ПК: 234.00

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_12")
df_data = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

summary = df_data.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

summary["duration_hours"] = summary["duration_hours"].round(2)
summary = summary.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(summary)

,game_title,total_paid,duration_hours
0,PUBG,26415,2.23
1,Valorant,26018,2.10
2,Counter-Strike 2,24997,2.12
3,Cyberpunk 2077,24616,2.14
4,Atomic Heart,23920,2.29
5,Rust,23126,2.05
6,Warzone,22818,2.01
7,Apex Legends,19695,2.15
8,Dota 2,16630,1.95
9,GTA V,16457,2.06


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` для ряда сессий значение в столбце `client_rating` равно `NULL`.
Чем для СУБД и для аналитика значение `NULL` принципиально отличается от оценки `0` при расчете среднего рейтинга `AVG(client_rating)`?

**Ответ:**
`NULL` означает отсутствие значения (клиент не оставил оценку), а `0` — это конкретная выставленная оценка. При вычислении `AVG(client_rating)` СУБД автоматически игнорирует строки со значением `NULL`, тогда как оценка `0` вошла бы в расчет и сильно занизила средний рейтинг клуба.

---

#### 3.2. Разработчик через скрипт на Python выполнил добавление новых компьютеров командой `cursor.execute("INSERT INTO hardware_pcs ...")`, но после закрытия программы обнаружил, что компьютеры не сохранились в файле базы данных.
Какую обязательную команду для объекта соединения (`conn`) он забыл вызвать и почему?

**Ответ:**
Он забыл вызвать команду фиксации транзакции `conn.commit()`. В библиотеке `sqlite3` изменения, внесенные через курсор, по умолчанию находятся во временной транзакции и не сохраняются на диске без явного `commit()`.

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT status, COUNT(*) AS pcs_cnt, AVG(ram_gb) AS avg_ram FROM hardware_pcs GROUP BY status;
```

**Ответ:**
Запрос группирует компьютеры по их рабочему статусу (`status`) и рассчитывает количество ПК и средний объем оперативной памяти в каждой категории.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_12")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT tournament_id, tournament_name, game_title, prize_fund, entry_fee, start_date
FROM tournaments
WHERE game_title IN ('Counter-Strike 2', 'Dota 2')
ORDER BY prize_fund DESC, start_date ASC;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,tournament_id,tournament_name,game_title,prize_fund,entry_fee,start_date
0,106,Pro Masters Elite: CS2,Counter-Strike 2,120000.0,1000.0,2026-05-01
1,102,Spring DOTA 2 Major,Dota 2,75000.0,700.0,2026-03-08
2,101,Winter Cyber Cup: CS2,Counter-Strike 2,50000.0,500.0,2026-02-14
3,107,Dota 2 Weekend Brawl,Dota 2,30000.0,350.0,2026-05-15


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT city, COUNT(*) AS members_count, ROUND(AVG(balance), 2) AS avg_balance
FROM members
GROUP BY city
HAVING AVG(balance) > 1000
ORDER BY avg_balance DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,city,members_count,avg_balance
0,Санкт-Петербург,35,2744.29
1,Казань,23,2363.04
2,Москва,67,1741.04
3,Новосибирск,15,1646.67
4,Екатеринбург,10,1460.00


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT m.member_id, m.nickname, COUNT(b.order_id) AS orders_count, ROUND(SUM(b.total_amount), 2) AS total_spent
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
WHERE m.city = 'Казань'
GROUP BY m.member_id, m.nickname
ORDER BY total_spent DESC
LIMIT 8;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,member_id,nickname,orders_count,total_spent
0,58,Falcon_129,5,5830.0
1,135,Frost_987,7,3490.0
2,57,Falcon_374,2,3000.0
3,86,Blaze_176,2,2280.0
4,105,Reaper_368,3,2240.0
5,95,CyberKnight_663,3,1930.0
6,102,Flash_459,3,1910.0
7,75,Slayer_289,3,1170.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE city = 'Москва' AND balance > (SELECT AVG(balance) FROM members WHERE city = 'Москва')
ORDER BY balance DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,member_id,nickname,city,loyalty_tier,balance
0,82,Wolf_446,Москва,Bronze,7500.0
1,120,Wolf_27,Москва,Bronze,7500.0
2,131,Paladin_338,Москва,Bronze,7500.0
3,5,Thunder_856,Москва,Platinum,4000.0
4,7,Phoenix_897,Москва,Gold,4000.0
5,40,Viper_439,Москва,Bronze,4000.0
6,42,Blade_169,Москва,Bronze,4000.0
7,60,Dragon_433,Москва,Bronze,4000.0
8,64,Valkyrie_973,Москва,Bronze,4000.0
9,92,Dragon_659,Москва,Gold,4000.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Список клиентов премиального уровня

**Найденные ошибки:**
1. Ошибка строкового литерала: текстовое значение `Platinum` указано без кавычек, из-за чего СУБД воспринимает его как имя несуществующего столбца.

In [9]:
query_51_fixed = '''
SELECT 
    member_id,
    nickname,
    city,
    loyalty_tier,
    balance
FROM members
WHERE loyalty_tier = 'Platinum'
ORDER BY balance DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,member_id,nickname,city,loyalty_tier,balance
0,25,Samurai_70,Санкт-Петербург,Platinum,7500.0
1,122,Slayer_12,Казань,Platinum,7500.0
2,5,Thunder_856,Москва,Platinum,4000.0
3,101,Valkyrie_893,Москва,Platinum,2500.0
4,21,Ranger_421,Казань,Platinum,1500.0
5,128,Matrix_139,Санкт-Петербург,Platinum,1000.0
6,79,Neo_285,Москва,Platinum,500.0
7,113,Valkyrie_49,Новосибирск,Platinum,500.0
8,137,Apex_761,Екатеринбург,Platinum,500.0
9,150,Shadow_996,Москва,Platinum,300.0


#### 5.2. Выручка по игровым местам

**Найденные ошибки:**
1. Ошибка ключа соединения: `p.pc_id = s.member_id` связывает компьютер с идентификатором клиента вместо `s.pc_id`.
2. Агрегатная функция в `WHERE`: условие `SUM(s.total_paid) > 5000` должно быть перенесено в блок `HAVING`.

In [10]:
query_52_fixed = '''
SELECT 
    p.pc_id,
    p.pc_number,
    p.zone_name,
    SUM(s.total_paid) AS pc_revenue
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.pc_id, p.pc_number, p.zone_name
HAVING SUM(s.total_paid) > 5000
ORDER BY pc_revenue DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,pc_id,pc_number,zone_name,pc_revenue
0,24,PC-24,VIP,20979.0
1,23,PC-23,VIP,16500.0
2,25,PC-25,VIP,13700.0
3,20,PC-20,Bootcamp,12363.0
4,26,PC-26,VIP,11928.0
5,18,PC-18,Bootcamp,11764.0
6,19,PC-19,Bootcamp,10163.0
7,21,PC-21,Bootcamp,8708.0
8,22,PC-22,Bootcamp,8625.0
9,9,PC-09,Standard,8430.0


#### 5.3. Сравнение выручки компьютеров со средней оплатой сессий

**Найденные ошибки:**
1. Порядок команд: блок `WHERE` размещен после `GROUP BY`, фильтрация агрегатных данных должна выполняться в `HAVING`.
2. Использование псевдонима: указан alias `avg_session_paid` вместо агрегата `AVG(s.total_paid)`.
3. Многострочный подзапрос: наличие `GROUP BY zone_name` заставляет подзапрос возвращать вектор значений вместо одного скалярного числа.

In [11]:
query_53_fixed = '''
SELECT 
    p.pc_id,
    p.pc_number,
    p.zone_name,
    ROUND(AVG(s.total_paid), 2) AS avg_session_paid
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.pc_id, p.pc_number, p.zone_name
HAVING AVG(s.total_paid) > (
    SELECT AVG(total_paid) 
    FROM game_sessions
)
ORDER BY avg_session_paid DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,pc_id,pc_number,zone_name,avg_session_paid
0,23,PC-23,VIP,916.67
1,25,PC-25,VIP,913.33
2,24,PC-24,VIP,839.16
3,26,PC-26,VIP,701.65
4,19,PC-19,Bootcamp,534.89
5,20,PC-20,Bootcamp,515.13
6,21,PC-21,Bootcamp,512.24
7,17,PC-17,Bootcamp,502.06
8,15,PC-15,Bootcamp,492.65
9,22,PC-22,Bootcamp,479.17
